<a href="https://colab.research.google.com/github/AgentForgeWorkshop/Tutorials/blob/main/Tutorial3_mcp_skills/03_mcp_skills_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3: Your own research MCP server (Colab version)

**AgentForge Workshop · Tutorial 3 · about 40 minutes**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentForgeWorkshop/Tutorials/blob/main/Tutorial3_mcp_skills/03_mcp_skills_colab.ipynb)

The main hands-on runs on your laptop, so that Claude Desktop, Claude Code, Cursor or VS Code can
use your server. **Use this notebook if the local install does not work for you.** It runs the
same server and the same code, with one difference: a desktop AI app cannot connect to Colab, so
the AI app here is `chat.py`, a small agent loop that talks to your server over MCP.

| Step | What you add |
|---|---|
| 1 | Search and read your own PDFs, with page citations |
| 2 | A cited review of published papers online |
| 3 | A calculator skill: exact arithmetic and the method to use it |
| 4 | A lay abstract that is written by AI and checked by code |
| 5 | Valency in ChatGPT: a hosted MCP server, no code |

Every step is also explained in the [`docs/`](https://github.com/AgentForgeWorkshop/Tutorials/tree/main/Tutorial3_mcp_skills/docs) folder.

---
## Step 0: Setup

Run the next cell once. It downloads the tutorial and installs it, in about a minute.

In [ ]:
import os
from pathlib import Path

if not Path("/content/Tutorials").exists():
    !git clone -q https://github.com/AgentForgeWorkshop/Tutorials.git /content/Tutorials
%cd /content/Tutorials/Tutorial3_mcp_skills
%pip install -q -r requirements.txt

### Your API key

Use the same key as in Tutorial 1:

1. Open <https://openrouter.ai/keys> and create a key (the workshop gives you credit).
2. In Colab, click the **🔑 key icon** in the left sidebar (Secrets).
3. Add a secret named exactly `OPENROUTER_API_KEY`, paste your key, and turn on **Notebook access**.

Optional: add a free OpenAlex key from <https://openalex.org/settings/api> as a secret named
`OPENALEX_API_KEY`. Without it, everyone on the same network shares one small daily search budget.

A key pasted into a cell gets saved into the notebook, and notebooks get shared. Treat an API key
like a password.

In [ ]:
import getpass


def get_key(name: str, required: bool = True) -> str:
    """Read an API key from Colab's Secrets panel, or ask for it if there is none."""
    try:
        from google.colab import userdata

        return userdata.get(name) or ""
    except Exception:  # not in Colab, or that secret is not set
        if not required:
            return os.environ.get(name, "")
        return os.environ.get(name) or getpass.getpass(f"{name}: ")


PROVIDER = "openrouter"  # <-- "openrouter" or "google"

if PROVIDER == "openrouter":
    os.environ["LLM_API_KEY"] = get_key("OPENROUTER_API_KEY")
elif PROVIDER == "google":
    os.environ["LLM_API_KEY"] = get_key("GOOGLE_API_KEY")
os.environ["OPENALEX_API_KEY"] = get_key("OPENALEX_API_KEY", required=False)

# Optional: any OpenRouter model that supports tool calling, for example "anthropic/claude-sonnet-5.5".
# os.environ["LLM_MODEL"] = "deepseek/deepseek-v4.1-flash"
print("Key ready:", "yes" if os.environ["LLM_API_KEY"] else "NO, add the secret and run this cell again")

---
## Step 1: Your local library

Your library is a folder of PDF, `.txt` or `.md` files. To use your own papers, open the **📁 Files**
panel on the left, then drag PDFs into the `my_papers` folder that the next cell creates. If the
folder is empty, the five practice papers in `sample_library/` are used instead. They are
synthetic, so do not cite them.

In [ ]:
LIBRARY = Path("/content/my_papers")  # <-- or any folder, for example on your Google Drive
LIBRARY.mkdir(exist_ok=True)

has_files = any(p.suffix.lower() in {".pdf", ".txt", ".md"} for p in LIBRARY.rglob("*"))
os.environ["LIBRARY_PATH"] = str(LIBRARY if has_files else Path("sample_library").resolve())
print("Library folder:", os.environ["LIBRARY_PATH"])

### Talk to the server, with no AI

This is what every AI app does when it connects: start the server, **discover** what it offers,
then **call** a tool. No model is involved, so it is free.

In [ ]:
!python try_server.py

In [ ]:
!python try_server.py local "stack pressure"

✅ **Checkpoint:** 11 tools, your documents, and passages labelled with a file and page.

### Now let an AI use it

`chat.py` is an AI app in about 100 lines: it gives the model the list of tools from MCP, runs the
tools the model asks for, and sends back the results. Watch the `[tool]` lines: each one is a
decision the model made.

In [ ]:
!python chat.py "Do my papers agree on stack pressure versus interlayers? Cite the file and page."

To see how a tool is written, open `server.py` in the Files panel and find `search_papers`. The
**docstring** is what the model reads when it decides whether to use the tool.

---
## Step 2: Online literature

Search needs no AI key. It asks OpenAlex, arXiv and Europe PMC at the same time.

In [ ]:
!python try_server.py online "lithium dendrite interlayer"

With your key, one tool writes a whole review. Only two of its four steps use AI: the model chooses
the searches and writes the review, while code searches the databases, checks every citation
number and builds the reference list from the database records. The model cannot invent a
reference, but it can misread an abstract, so open two of the papers and check.

In [ ]:
!python try_server.py review "Do interlayers suppress lithium dendrites at high stack pressure?"

---
## Step 3: Calculator skill

A **tool** gives the ability: `calculate` does exact arithmetic and refuses anything that is not
maths. A **skill** gives the method: `skills/calculator/SKILL.md` says when to calculate, which
formula to use and how to report it. The server serves skills in stages with `list_skills`,
`load_skill` and `read_skill_file`.

In [ ]:
!python try_server.py calc "412/487*100"
!python try_server.py calc "__import__('os')"

In [ ]:
!python chat.py "A cell keeps 412 of its 487 mAh after 500 cycles. What are the retention and the fade per cycle?"

✅ **Checkpoint:** `calculate` runs at each step, giving 84.6% and about 0.031% per cycle.

---
## Step 4: Lay abstract review

The AI writes the plain-language summary, then `review_lay_abstract`, which is ordinary code,
measures it: word count, reading level, long sentences, possible jargon, and **any number that
does not appear in the source**.

In [ ]:
!python try_server.py lay paper5

In [ ]:
!python chat.py "Write a lay abstract of paper5 for first-year students."

The checker measures readability and matches numbers. It cannot tell whether the summary is
*right*. Read it against the paper: is this the main finding, and did "suggests" stay "suggests"?

---
## Step 5: Valency in ChatGPT

The last step needs no code. Your server runs on your machine or here in Colab. Valency's runs on
the internet, so ChatGPT can connect to it. Follow
[docs/05-valency-chatgpt.md](https://github.com/AgentForgeWorkshop/Tutorials/blob/main/Tutorial3_mcp_skills/docs/05-valency-chatgpt.md).

---
## Wrap-up

* **MCP gives access.** You wrapped plain Python functions as tools, and an AI app discovered them by itself.
* **Skills give know-how.** A folder with a `SKILL.md` tells the agent how to do a job well.
* **Keep the checkable facts in code.** References from database records, a citation check, a
  number check. The model gets the judgement, and code keeps it honest.

To use your server in Claude Desktop, Claude Code, Cursor or VS Code, install it on your laptop:
see the [README](https://github.com/AgentForgeWorkshop/Tutorials/tree/main/Tutorial3_mcp_skills).

---
*AgentForge Workshop, Tutorial 3. Questions, fixes and better examples are welcome:
<https://github.com/AgentForgeWorkshop/Tutorials>*